In [1]:
pip install pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. LOAD DATA
# ============================================================

menu_file = "menu.csv"
history_file = "order_history.csv"

menu = pd.read_csv(menu_file)
history = pd.read_csv(history_file)

print("========================================")
print("      FOOD DELIVERY RAG CHATBOT")
print("========================================")


# ============================================================
# 2. CLEAN DATA
# ============================================================

menu["available"] = menu["available"].astype(str).str.lower()
menu["diet"] = menu["diet"].astype(str).str.lower()
menu["spicy"] = menu["spicy"].astype(str).str.lower()

# Create searchable text
menu["search_text"] = (
    menu["dish"].astype(str) + " " +
    menu["restaurant"].astype(str) + " " +
    menu["category"].astype(str) + " " +
    menu["description"].astype(str) + " " +
    menu["diet"].astype(str) + " " +
    menu["spicy"].astype(str)
)


# ============================================================
# 3. CREATE RAG RETRIEVER
# ============================================================

vectorizer = TfidfVectorizer(stop_words="english")

menu_vectors = vectorizer.fit_transform(menu["search_text"])


# ============================================================
# 4. GET USER HISTORY
# ============================================================

def get_user_history(user_id):

    user_orders = history[
        history["user_id"].astype(str).str.upper() == user_id.upper()
    ]

    if user_orders.empty:
        return []

    return user_orders["dish"].tolist()


# ============================================================
# 5. RETRIEVE RELEVANT MENU ITEMS
# ============================================================

def retrieve_menu(query, top_k=5):

    query_vector = vectorizer.transform([query])

    similarity = cosine_similarity(
        query_vector,
        menu_vectors
    ).flatten()

    # Sort by similarity
    indexes = similarity.argsort()[::-1]

    results = []

    for index in indexes:

        # Only recommend currently available dishes
        if menu.iloc[index]["available"] == "yes":

            results.append(menu.iloc[index])

        if len(results) == top_k:
            break

    return pd.DataFrame(results)


# ============================================================
# 6. FILTER BY DIETARY RESTRICTION
# ============================================================

def filter_diet(results, query):

    query = query.lower()

    # Vegetarian
    if "vegetarian" in query or "veg" in query:
        results = results[
            results["diet"] == "vegetarian"
        ]

    # Non-vegetarian
    elif "non vegetarian" in query or "non-vegetarian" in query:
        results = results[
            results["diet"] == "non-vegetarian"
        ]

    return results


# ============================================================
# 7. CHECK SPICY PREFERENCE
# ============================================================

def filter_spicy(results, query):

    query = query.lower()

    if "not spicy" in query or "non spicy" in query:

        results = results[
            results["spicy"] == "no"
        ]

    elif "spicy" in query:

        results = results[
            results["spicy"] == "yes"
        ]

    return results


# ============================================================
# 8. PERSONALIZED RECOMMENDATION
# ============================================================

def recommend(user_id, query):

    # Get previous orders
    previous_orders = get_user_history(user_id)

    # Retrieve menu using RAG
    results = retrieve_menu(query, top_k=10)

    # Apply dietary filter
    results = filter_diet(results, query)

    # Apply spicy preference
    results = filter_spicy(results, query)

    if results.empty:

        print("\nSorry, no suitable dishes are currently available.")
        return

    print("\n========================================")
    print("PERSONALIZED RECOMMENDATIONS")
    print("========================================")

    if previous_orders:

        print("\nYour previous orders:")

        for dish in previous_orders:
            print("-", dish)

    else:

        print("\nNo previous order history found.")

    print("\nRecommended dishes:\n")

    # Show maximum 3 recommendations
    for _, row in results.head(3).iterrows():

        print(f"🍽️ {row['dish']}")
        print(f"   Restaurant : {row['restaurant']}")
        print(f"   Category   : {row['category']}")
        print(f"   Price      : ₹{row['price']}")
        print(f"   Diet       : {row['diet']}")
        print(f"   Spicy      : {row['spicy']}")
        print(f"   Available  : {row['available']}")
        print(f"   Reason     : {row['description']}")
        print()


# ============================================================
# 9. CHAT LOOP
# ============================================================

while True:

    print("\n----------------------------------------")

    user_id = input("Enter User ID (U001/U002/U003): ").strip()

    if user_id.lower() == "exit":
        print("Goodbye!")
        break

    query = input(
        "What would you like to eat? "
        "(type 'exit' to quit): "
    ).strip()

    if query.lower() == "exit":
        print("Goodbye!")
        break

    recommend(user_id, query)

      FOOD DELIVERY RAG CHATBOT

----------------------------------------

PERSONALIZED RECOMMENDATIONS

Your previous orders:
- Paneer Tikka
- Veg Biryani
- Masala Dosa

Recommended dishes:

🍽️ Veg Biryani
   Restaurant : Biryani House
   Category   : Indian
   Price      : ₹200
   Diet       : vegetarian
   Spicy      : yes
   Available  : yes
   Reason     : Aromatic rice with vegetables and spices

🍽️ Paneer Tikka
   Restaurant : Spice Garden
   Category   : Indian
   Price      : ₹220
   Diet       : vegetarian
   Spicy      : yes
   Available  : yes
   Reason     : Grilled paneer with Indian spices

🍽️ Veg Hakka Noodles
   Restaurant : Chinese Wok
   Category   : Chinese
   Price      : ₹190
   Diet       : vegetarian
   Spicy      : yes
   Available  : yes
   Reason     : Stir fried noodles with vegetables


----------------------------------------

PERSONALIZED RECOMMENDATIONS

Your previous orders:
- Chicken Biryani
- Chicken Noodles

Recommended dishes:

🍽️ Veg Biryani
   Res